In [24]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [25]:
with open('data\paul_graham_essays.txt', 'r') as file:
    data = '\n'.join(line.strip() for line in file if line.strip())[:50000]

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Nitin\AppData\Local\Temp\ipykernel_32272\2941056223.py:1: SyntaxWarning: invalid escape sequence '\p'
  with open('data\paul_graham_essays.txt', 'r') as file:


In [26]:
from calendar import c

vocab_size = len(set(data))
total_size = len(data)
print(f"Total size of the text: {total_size} characters")
print(f"Vocabulary size: {vocab_size}")

char_to_index = {ch: i for i, ch in enumerate(sorted(set(data)))}
index_to_char = {i: ch for i, ch in enumerate(sorted(set(data)))}

print(char_to_index.keys())

Total size of the text: 50000 characters
Vocabulary size: 77
dict_keys([' ', '!', '"', "'", '(', ')', '+', ',', '-', '.', '0', '1', '2', '3', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\x97', '\xa0', '—'])


In [ ]:
from matplotlib.pyplot import bar


class RNN:
    def __init__(self, vocab_size, context_window=25, hidden_size=200):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.context_window = context_window
        self._init_weights()
    
    def _init_weights(self):
        self.Whh = np.random.randn(self.hidden_size, self.hidden_size) * (2 / np.sqrt(self.hidden_size))
        self.Wxh = np.random.randn(self.hidden_size, self.vocab_size) * (2 / np.sqrt(self.vocab_size))
        self.Why = np.random.randn(self.vocab_size, self.hidden_size) * (2 / np.sqrt(self.hidden_size))
        self.bh = np.zeros((self.hidden_size, 1))
        self.by = np.zeros((self.vocab_size, 1))

    def forward(self, input, output):
        '''
        Forward pass through the RNN. 
        Goes through the context window in X and computes the hidden state, outputs, gradients and 
        loss.
        X: input data
        y: target data
        '''
        # Convert input and output strings to one-hot encoded representations
        X = np.zeros((self.context_window, self.vocab_size))
        Y = np.zeros((self.context_window, self.vocab_size))
        for i, char in enumerate(input):
            X[i, char_to_index[char]] = 1
        for i, char in enumerate(output):
            Y[i, char_to_index[char]] = 1
            
        h = [np.zeros((self.hidden_size, 1)) for _ in range(self.context_window)]
        y = [np.zeros((self.vocab_size, 1)) for _ in range(self.context_window)]
        loss = 0.0
        for t in range(self.context_window):
            # Update hidden state: 
            # (hidden_size, 1) = (hidden_size, hidden_size) @ (hidden_size, 1) + (hidden_size, vocab_size) @ (vocab_size, 1) + (hidden_size, 1)
            h[t] = np.tanh(np.dot(self.Whh, h[t - 1]) + np.dot(self.Wxh, X[t].reshape(-1, 1)) + self.bh)
            
            # Compute output: 
            # (vocab_size, 1) = (vocab_size, hidden_size) @ (hidden_size, 1) + (vocab_size, 1)
            y[t] = np.dot(self.Why, h[t]) + self.by

            y_shifted = y[t] - np.max(y[t])  # for numerical stability
            y_softmax = np.exp(y_shifted) / np.sum(np.exp(y_shifted))
            outchar = index_to_char[np.argmax(y_softmax)]
            # print(f"Output at time step {t}: {outchar}")

            # Compute loss: scalar value
            loss += -np.sum(Y[t].reshape(-1, 1) * np.log(y_softmax + 1e-8))

        # Gradients initialization
        # Initialize gradients for weights and biases.
        d_Whh = np.zeros_like(self.Whh)
        d_Wxh = np.zeros_like(self.Wxh)
        d_Why = np.zeros_like(self.Why)
        d_bh = np.zeros_like(self.bh)
        d_by = np.zeros_like(self.by)

        dh = np.zeros_like(h[0])

        # Backward pass through the RNN.
        for t in reversed(range(self.context_window)):
            y_pred = np.exp(y[t]) / np.sum(np.exp(y[t]))
            h_next = h[t]
            h_prev = h[t - 1] if t > 0 else np.zeros((self.hidden_size, 1))

            # Compute gradients
            # Gradient of loss w.r.t. input to softmax + cross-entropy log loss.
            d_y = y_pred - Y[t].reshape(-1, 1)
            # Gradient of loss w.r.t. output layer weights and biases.
            d_Why += np.dot(d_y, h_next.T)
            d_by += d_y

            # Gradient of loss w.r.t. hidden state.
            dh += np.dot(self.Why.T, d_y)
            dh_raw = (1 - h_next ** 2) * dh

            d_Whh += np.dot(dh_raw, h_prev.T)
            d_Wxh += np.dot(dh_raw, X[t].reshape(1, -1))
            d_bh += dh_raw

            dh = np.dot(self.Whh.T, dh_raw)  # pass to next timestep

        for grad in [d_Whh, d_Wxh, d_Why, d_bh, d_by]:
            np.clip(grad, -15, 15, out=grad)

        loss /= self.context_window

        return loss, d_Whh, d_Wxh, d_Why, d_bh, d_by

    def _update_weights_with_adam(self, grad, learning_rate):
        # Unpack gradients
        d_Whh, d_Wxh, d_Why, d_bh, d_by = grad

        # Initialize Adam optimizer parameters
        if not hasattr(self, 'grad'):
            self.grad = {
            'm': {
                'Whh': np.zeros_like(self.Whh), 
                'Wxh': np.zeros_like(self.Wxh), 
                'Why': np.zeros_like(self.Why), 
                'bh': np.zeros_like(self.bh), 
                'by': np.zeros_like(self.by)
            },
            'v': {
                'Whh': np.zeros_like(self.Whh), 
                'Wxh': np.zeros_like(self.Wxh), 
                'Why': np.zeros_like(self.Why), 
                'bh': np.zeros_like(self.bh), 
                'by': np.zeros_like(self.by)
            }}
            self.t = 0  # Time step

        # Update time step
        self.t += 1
        beta1, beta2, epsilon = 0.9, 0.999, 1e-8

        # Update biased first moment estimate
        for key, grad_value in zip(['Whh', 'Wxh', 'Why', 'bh', 'by'], grad):
            self.grad['m'][key] = beta1 * self.grad['m'][key] + (1 - beta1) * grad_value

            # Update biased second raw moment estimate
            self.grad['v'][key] = beta2 * self.grad['v'][key] + (1 - beta2) * (grad_value ** 2)

            # Compute bias-corrected first and second moment estimates
            m_hat = self.grad['m'][key] / (1 - beta1 ** self.t)
            v_hat = self.grad['v'][key] / (1 - beta2 ** self.t)

            # Update weights and biases
            setattr(self, key, getattr(self, key) - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon))


    def train(self, learning_rate=0.01, epochs=1000, decay_rate=0.95, grad_norm_clip=5.0):
        initial_learning_rate = learning_rate
        
        for epoch in range(epochs):
            loss_epoch = 0.0
            for batch_idx in range(len(data) - self.context_window):
                input_seq = data[batch_idx:batch_idx + self.context_window]
                output_seq = data[batch_idx + 1:batch_idx + self.context_window + 1]
                loss, d_Whh, d_Wxh, d_Why, d_bh, d_by = self.forward(input_seq, output_seq)

                # Monitor gradient norms
                total_norm = np.sqrt(
                    np.sum([np.sum(grad ** 2) for grad in [d_Whh, d_Wxh, d_Why, d_bh, d_by]])
                )
                if total_norm > grad_norm_clip:
                    # if total_norm > grad_norm_clip:
                    #     print(f"[Warning] Large gradient norm detected: {total_norm:.2f} at Epoch {epoch} Batch {batch_idx}")         
                    clip_coef = grad_norm_clip / (total_norm + 1e-6)
                    d_Whh *= clip_coef
                    d_Wxh *= clip_coef
                    d_Why *= clip_coef
                    d_bh *= clip_coef
                    d_by *= clip_coef
                
                self._update_weights_with_adam([d_Whh, d_Wxh, d_Why, d_bh, d_by], learning_rate)
                loss_epoch += loss

                if batch_idx % 1000 == 0:
                    perc = batch_idx * 100 / (len(data) - self.context_window)
                    print(f"Epoch({epoch}) {perc:.2f}%, Loss per character: {loss:.4f}")

            # Decay the learning rate after each epoch
            learning_rate = initial_learning_rate * (decay_rate ** (epoch + 1))
            print(f"Epoch({epoch}) completed. Avg Loss: {loss_epoch / (len(data) - self.context_window):.4f}, New LR: {learning_rate:.6f}")

    def inference(self, start_text, length=100, temperature=1.0):
        '''
        Generate text from the trained RNN model.
        
        Args:
            start_text (str): Initial prompt to start generation.
            length (int): Total number of characters to generate.
            temperature (float): Sampling temperature (>1: more random, <1: more greedy).

        Returns:
            generated_text (str): Generated sequence including start_text.
        '''
        # Initialize hidden states
        h = np.zeros((self.hidden_size, 1))
        
        # Prepare start sequence
        input_seq = [char_to_index[ch] for ch in start_text]
        generated_text = start_text

        # Feed the start text first
        for idx in input_seq[:-1]:
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1
            h = np.tanh(np.dot(self.Whh, h) + np.dot(self.Wxh, x) + self.bh)

        # Now, generate further characters
        idx = input_seq[-1]
        for _ in range(length):
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1

            h = np.tanh(np.dot(self.Whh, h) + np.dot(self.Wxh, x) + self.bh)
            y = np.dot(self.Why, h) + self.by
            
            # Apply temperature
            y = y / temperature
            
            # Softmax
            y_exp = np.exp(y - np.max(y))  # stabilize
            probs = y_exp / np.sum(y_exp)
            
            # Sample next character index
            idx = np.random.choice(range(self.vocab_size), p=probs.ravel())
            next_char = index_to_char[idx]
            
            generated_text += next_char

        return generated_text



In [ ]:
model = RNN(vocab_size, context_window=25, hidden_size=100)
model.train(learning_rate=0.005, epochs=1, grad_norm_clip=15.0)


Epoch(0) 0.00%, Loss per character: 5.5804
Epoch(0) 2.00%, Loss per character: 2.5134
Epoch(0) 4.00%, Loss per character: 2.3759
Epoch(0) 6.00%, Loss per character: 2.1821
Epoch(0) 8.00%, Loss per character: 2.2612
Epoch(0) 10.01%, Loss per character: 2.3197
Epoch(0) 12.01%, Loss per character: 2.6953
Epoch(0) 14.01%, Loss per character: 2.0672
Epoch(0) 16.01%, Loss per character: 2.8147
Epoch(0) 18.01%, Loss per character: 2.9671
Epoch(0) 20.01%, Loss per character: 3.1337
Epoch(0) 22.01%, Loss per character: 2.4108
Epoch(0) 24.01%, Loss per character: 2.8815
Epoch(0) 26.01%, Loss per character: 2.9396
Epoch(0) 28.01%, Loss per character: 2.5295
Epoch(0) 30.02%, Loss per character: 2.8398
Epoch(0) 32.02%, Loss per character: 2.2714
Epoch(0) 34.02%, Loss per character: 2.7058
Epoch(0) 36.02%, Loss per character: 2.8735
Epoch(0) 38.02%, Loss per character: 2.1940
Epoch(0) 40.02%, Loss per character: 1.8766
Epoch(0) 42.02%, Loss per character: 2.8989
Epoch(0) 44.02%, Loss per character: 

'The right approach to inveryeirdrllile dehuorty idhehhitr thhit roy iseythee yir ioel srrianrsa hrarlertit lrtinoetdy thehhrlo'

In [34]:
model.inference("The right approach to inve", length=100, temperature=1.0)

'The right approach to inveersee itil ids ise se arneierrere hl reere hont hdeeiier isee ied s rt thhane ide thrlyr hil drreisy'